In [1]:
import sys
sys.path.append("/scratch/mzaffar/olaf/VPR-methods-evaluation/")
import torch
import numpy as np
import os
import datasets_ws
import matplotlib.pyplot as plt
import faiss
import gc
from os.path import join
from glob import glob
from tqdm import tqdm
from parser import parse_arguments
from sklearn.decomposition import PCA
from vpr_models import get_model
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Subset
import torchvision.transforms as transforms

from utils import *

own database ws


In [2]:
model_folder="/scratch/mzaffar/olaf/weights/finetuned/"
base_models_folder="/scratch/mzaffar/olaf/weights/"
models_pths=sorted(glob(join(model_folder, "**", "*.pth"), recursive=True))
args=parse_arguments(["--method_pths","gsv_crica","gsv_boq", #"gsv_boq_finetuned_on_nordland.pth","gsv_crica_finetuned_on_nordland.pth",
                      "--method_folder","weights/finetuned",
                      "--datasets_folder","../datasets_vg/datasets/",
                      "--dataset_name","nordland",
                      "--batch_size","64",
                      "--gpu_id","3"
                    ])
os.environ["CUDA_VISIBLE_DEVICES"]=args.gpu_id
models_pths=[join(args.method_folder,method) for method in args.method_pths]

In [3]:
for pth in models_pths:
    #determining which model to initialize from path name
    name=pth.split(".")[0].split("/")[-1]
    model_type=name.split("_")[1]
    trainings_dataset_name=name.split("_")[0]
    finetune_dataset_name=name.split("_")[-1]
    print(name)

gsv_crica
gsv_boq


In [12]:
class Method():
    def __init__(self,args,model_pth):
        pth_small=pth.split(".")[0].split("/")[-1]
        self.name=pth_small.split("_")[1]
        self.trainings_dataset_name=pth_small.split("_")[0]
        self.finetune_dataset_name=pth_small.split("_")[-1]
        if self.name==self.finetune_dataset_name:
            model_pth=join(base_models_folder,pth_small+".pth")
            print(f"Loading method {self.name} not finetuned")
        else:
            print(f"Loading method {self.name} finetuned on {self.finetune_dataset_name}")
            
        self.model=get_model(args,self.name,model_pth)
        self.features_dim=args.features_dim
        
        self.resize=args.image_size
        self.sue_scores=None
        self.predictions=None
        self.distances=None
        self.corrects_per_query=None
        self.recalls=None
        
    def evaluate(self,args,eval_ds):
        print(f"evaluating method {self.name} on dataset {eval_ds.dataset_name}")
        eval_ds.resize=self.resize
        model,rerank=self.model
        model.cuda()
        if args.pca==True:
            pca=self.pca
        else:
            pca=None
        model = model.eval()
        with torch.no_grad():
            print("Extracting database features for evaluation/testing")
            # For database use "hard_resize", although it usually has no effect because database images have same resolution
            eval_ds.test_method = "hard_resize"
            database_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num)))
            database_dataloader = DataLoader(dataset=database_subset_ds, num_workers=args.num_workers,
                                            batch_size=args.batch_size, pin_memory=(args.device=="cuda"))

            all_features = np.empty((len(eval_ds), self.features_dim), dtype="float32")
            if rerank!=None:
                W, H, C = args.dense_feature_map_size
                all_local_features = np.empty((len(eval_ds), W, H, C), dtype="float32")

            for inputs, indices in tqdm(database_dataloader, ncols=100):
                
                if rerank!=None:
                    local_features, features = model(inputs.to(args.device))
                elif self.name=='boq':
                    features,_=model(inputs.to(args.device))
                else:
                    features = model(inputs.to(args.device))
                    
                features = features.cpu().numpy()
                
                if pca != None:
                    features = pca.transform(features)
                all_features[indices.numpy(), :] = features
                if rerank!=None:
                    local_features = local_features.cpu().numpy()
                    all_local_features[indices.numpy(), :] = local_features

            print("Extracting queries features for evaluation/testing")
            eval_ds.test_method = "hard_resize"
            test_method=eval_ds.test_method
            queries_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num, eval_ds.database_num+eval_ds.queries_num)))
            queries_dataloader = DataLoader(dataset=queries_subset_ds, num_workers=args.num_workers,
                                            batch_size=args.batch_size, pin_memory=(args.device=="cuda"))
            
            for inputs, indices in tqdm(queries_dataloader, ncols=100):
                
                if test_method == "five_crops" or test_method == "nearest_crop" or test_method == 'maj_voting':
                    inputs = torch.cat(tuple(inputs))  # shape = 5*bs x 3 x 480 x 480
                
                if rerank!=None:
                    local_features, features = model(inputs.to(args.device))
                elif self.name=='boq':
                    features,_=model(inputs.to(args.device))
                else:
                    features = model(inputs.to(args.device))
                
                if test_method == "five_crops":  # Compute mean along the 5 crops
                    features = torch.stack(torch.split(features, 5)).mean(1)
                
                features = features.cpu().numpy()
                
                if pca != None:
                    features = pca.transform(features)
                all_features[indices.numpy(), :] = features
                
                if rerank!=None:
                    local_features = local_features.cpu().numpy()
                    all_local_features[indices.numpy(), :] = local_features
                
                if test_method == "nearest_crop" or test_method == 'maj_voting':  # store the features of all 5 crops
                    start_idx = eval_ds.database_num + (indices[0] - eval_ds.database_num) * 5
                    end_idx   = start_idx + indices.shape[0] * 5
                    indices = np.arange(start_idx, end_idx)
                    all_features[indices, :] = features
                else:
                    all_features[indices.numpy(), :] = features
                    if rerank!=None:
                        all_local_features[indices.numpy(), :] = local_features

        queries_features = all_features[eval_ds.database_num:]
        database_features = all_features[:eval_ds.database_num]
        if rerank!=None:
            queries_local_features = all_local_features[eval_ds.database_num:]
            database_local_features = all_local_features[:eval_ds.database_num]

        faiss_index = faiss.IndexFlatL2(self.features_dim)
        faiss_index.add(database_features)
        del database_features, all_features

        print("Calculating recalls")
        self.faiss=faiss_index
        
    
        distances, predictions = faiss_index.search(queries_features,args.n_features)

        #### For each query, check if the predictions are correct
        positives_per_query = eval_ds.get_positives()
        # args.recall_values by default is [1, 5, 10, 20]
        recalls = np.zeros(len(args.recall_values))
        for query_index, pred in enumerate(predictions):
            for i, n in enumerate(args.recall_values):
                if np.any(np.in1d(pred[:n], positives_per_query[query_index])):
                    recalls[i:] += 1
                    break
        # Divide by the number of queries*100, so the recalls are in percentages
        recalls = recalls / eval_ds.queries_num * 100
        recalls_str =", ".join([f"R@{val}: {rec:.1f}" for val, rec in zip(args.recall_values, recalls)])

        if rerank!=None:
            print(f"First ranking recalls: {recalls_str}")
            predictions = rerank(predictions,queries_local_features,database_local_features)

            #### For each query, check if the predictions are correct
            positives_per_query = eval_ds.get_positives()

            recalls = np.zeros(len(args.recall_values))
            for query_index, pred in enumerate(predictions):
                for i, n in enumerate(args.recall_values):
                    if np.any(np.in1d(pred[:n], positives_per_query[query_index])):
                        recalls[i:] += 1
                        break
            # Divide by the number of queries*100, so the recalls are in percentages
            recalls = recalls / eval_ds.queries_num * 100
            recalls_str = ", ".join([f"R@{val}: {rec:.1f}" for val, rec in zip(args.recall_values, recalls)])
            
        self.distances=distances
        self.predictions=predictions
        self.recalls=recalls
        self.corrects_per_query=positives_per_query
        self.ordered_distances=distances[np.arange(0,eval_ds.queries_num)[:,None],np.argsort(self.predictions)]
        del model, rerank, self.model
        print(f"resulting recalls of method {self.name}: {recalls_str}")
        return distances, predictions
    
    def compile_sue(self,ref_poses, num_NN=20, slope=350):
        self.sue_scores = np.zeros(len(self.predictions))

        print(f'Computing SUE uncertainty for method {self.name}')    
        weights = np.ones(num_NN)
        for itr in tqdm(range(len(self.sue_scores))):   
            top_preds = self.predictions[itr][:num_NN]
            nn_poses = ref_poses[top_preds]
            bm_pose = nn_poses[0]

            for itr2 in range(num_NN):
                weights[itr2] = math.e ** ((-1*abs(self.distances[itr][itr2])) * slope) 

            weights = weights/sum(abs(weights))

            mean_pose = np.asarray([np.average(nn_poses[:,0], weights=weights), np.average(nn_poses[:,1], weights=weights)])

            variance_lat_lat = 0 
            variance_lon_lon = 0    
            variance_lat_lon = 0    

            for k in range(0, num_NN):                
                diff_lat_lat = min(500, nn_poses[k,0] - mean_pose[0]) # so everything that is more than 500 meters away contributes equally to the variance 
                diff_lon_lon = min(500, nn_poses[k,1] - mean_pose[1])
                diff_lat_lon = min(500, nn_poses[k,0] - mean_pose[0]) *  min(500, nn_poses[k,1] - mean_pose[1])

                variance_lat_lat = variance_lat_lat + weights[k] * (diff_lat_lat)**2
                variance_lon_lon = variance_lon_lon + weights[k] * (diff_lon_lon)**2
                variance_lat_lon = variance_lat_lon + weights[k] * diff_lat_lon

            self.sue_scores[itr] = (variance_lat_lat + variance_lon_lon)/2  # assuming independent dimensions

        # sue_scores = -1 * sue_scores # converting into a confidence instead of an uncertainty
        # sue_scores_normalized = np.interp(sue_scores, (sue_scores.min(), sue_scores.max()), (0.0, 0.9999)) # avoiding infinity
        print('Done!') 
        return self.sue_scores
   

val_ds=datasets_ws.BaseDataset_normal(args,datasets_folder="../datasets_vg/datasets",dataset_name=args.dataset_name,split='test')

In [13]:
def combine_methods(args,methods_list):
    
    if args.fuse_method=="avg":
        final_dists=np.zeros([args.queries_num,args.database_num])
        #combining distances
        for method in methods_list:
            method_dists=np.zeros([args.queries_num,args.database_num])
            method_dists[np.arange(args.queries_num)[:,None],method.predictions]+=method.distances
            method_dists[method_dists==0]=np.max(method_dists)
            final_dists+=method_dists
        
        
            
            
            

In [14]:
methods_list=[]
for pth in models_pths:
    methods_list.append(Method(args,pth))
for method in methods_list:
    method.evaluate(args,val_ds)
    method.compile_sue(val_ds.database_utms)

Loading method crica not finetuned


Using cache found in /home/osverburg/.cache/torch/hub/Lu-Feng_CricaVPR_main


Loading method boq not finetuned


Using cache found in /home/osverburg/.cache/torch/hub/amaralibey_bag-of-queries_main
Using cache found in /home/osverburg/.cache/torch/hub/facebookresearch_dinov2_main


evaluating method crica on dataset nordland
Extracting database features for evaluation/testing


...

Extracting queries features for evaluation/testing


100%|█████████████████████████████████████████████████████████████| 432/432 [04:56<00:00,  1.46it/s]


Calculating recalls
resulting recalls of method boq: R@1: 90.4, R@5: 95.9, R@10: 97.4, R@20: 98.5
Computing SUE uncertainty for method boq


100%|██████████| 27592/27592 [00:05<00:00, 4923.28it/s]

Done!


In [ ]:
total_dists= np.zeros([val_ds.queries_num,val_ds.database_num])
for method in methods_list:
    total_dists[np.arange(val_ds.queries_num)[:,None],method.predictions]+=method.distances

In [ ]:
np.save("methods.npy",methods_list)
methods_list=np.load("methods.npy",allow_pickle=True)

In [ ]:
print(methods_list[0].ordered_distances.shape)

In [ ]:
dists_0=np.zeros([val_ds.queries_num,val_ds.database_num])
dists_1=np.zeros([val_ds.queries_num,val_ds.database_num])
dists_0[np.arange(val_ds.queries_num)[:,None],methods_list[0].predictions]+=methods_list[0].distances
dists_1[np.arange(val_ds.queries_num)[:,None],methods_list[1].predictions]+=methods_list[1].distances
# dists_0[dists_0==0]=1000
dists_0[dists_0==0]=np.max(dists_0)
dists_1[dists_1==0]=np.max(dists_1)
total_dists=dists_0+dists_1
total_dists[total_dists==0]=1000

In [ ]:
x=np.argsort(dists_0)[:,:100]

In [ ]:
y=np.sort(dists_0)[:100]

In [ ]:
print(y)
print(methods_l)

In [ ]:
print(x,x.shape)
print(methods_list[0].predictions,methods_list[0].predictions.shape)

In [ ]:
# print(total_dists)
# total_dists[total_dists==0]=2*np.max(total_dists)
def recalls(dists, corrects,recall_values=[1,5,10,20]):
    predictions=np.argsort(dists)#[:100]
    # predictions=methods_list[1].predictions
    positives_per_query=corrects
    recalls=np.zeros(len(args.recall_values))
    for query_index, pred in enumerate(predictions):
        for i, n in enumerate(args.recall_values):
            if np.any(np.in1d(pred[:n], positives_per_query[query_index])):
                recalls[i:] += 1
                break
    return recalls/len(corrects)
print(recalls(total_dists,methods_list[0].corrects_per_query))
print(recalls(dists_0,methods_list[0].corrects_per_query))
print(recalls(dists_1,methods_list[0].corrects_per_query))


In [ ]:
print((predictions[:,:100]==methods_list[0].predictions).sum()-(14278*100))

In [ ]:
print(recalls)

In [ ]:
preds=boq.predictions.copy()
dists=boq.distances.copy()
np.save("preds.npy",preds)
np.save("dists.npy",dists)
print(preds.shape,dists.shape,val_ds.queries_num)


In [ ]:
boq.model=None
val_ds=None
print(gc.collect())

with torch.no_grad():
   torch.cuda.empty_cache()

In [ ]:
x=np.stack([np.random.random(size=args.features_dim).astype(np.float32),np.random.random(size=args.features_dim).astype(np.float32)])
print(type(x[0,0]))
y=np.zeros([val_ds.queries_num,val_ds.database_num])
print(y.shape)

In [ ]:

y=np.zeros([val_ds.queries_num,val_ds.database_num])
print(y)
y[np.arange(val_ds.queries_num)[:,None],preds]+=dists
print(y)

In [ ]:
del boq.faiss
torch.cuda.empty_cache()

In [ ]:
x=np.linspace(0,val_ds.database_num-1,val_ds.database_num)
plt.scatter(x,y[0])